# 🤖 GenAI-Based Customer Support Chatbot V2
**Built with DialoGPT + Gradio | Real Multi-Turn Conversation**

This version maintains full conversation history so the bot replies
like a real human — remembering what was said before.

**Author:** Syed Mohammad Shahzeb  
**Skills Used:** Python, Hugging Face, Transformers, Prompt Engineering, Gradio

## Step 1: Install Required Libraries

In [ ]:
!pip install transformers gradio torch --quiet
print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


## Step 2: Import Libraries

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 3: Load DialoGPT Model

We use **`microsoft/DialoGPT-medium`** — a model specifically trained
for **multi-turn conversations**. Unlike regular models, it understands
the flow of a real chat.

In [ ]:
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading model: {MODEL_NAME} ... (this may take 1-2 minutes)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"✅ Model loaded on: {device}")

⏳ Loading model: microsoft/DialoGPT-medium ... (this may take 1-2 minutes)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model loaded on: cpu


## Step 4: Build the Conversation Function

This is the KEY part — we pass the **entire chat history** to the model
every time the user sends a message. This is what makes it a real chatbot.

In [ ]:
def generate_response(user_message, chat_history_ids=None):
    """
    Generates a reply from the chatbot.

    - user_message     : The new message typed by the user
    - chat_history_ids : All previous conversation tokens (memory)

    Returns:
    - bot_response      : The chatbot's reply text
    - new_history_ids   : Updated conversation history for next turn
    """

    # Step 1: Encode the new user message + end-of-string token
    new_input_ids = tokenizer.encode(
        user_message + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # Step 2: Append new message to the full conversation history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Step 3: Keep conversation within token limit (avoid overflow)
    max_history_tokens = 800
    if bot_input_ids.shape[-1] > max_history_tokens:
        bot_input_ids = bot_input_ids[:, -max_history_tokens:]

    # Step 4: Generate the bot's reply
    with torch.no_grad():
        new_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,          # Makes replies feel natural, not robotic
            top_k=50,                # Picks from top 50 likely words
            top_p=0.92,              # Nucleus sampling for better quality
            temperature=0.75         # Controls creativity (lower = focused)
        )

    # Step 5: Decode only the NEW part (bot's reply, not the whole history)
    bot_response = tokenizer.decode(
        new_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return bot_response, new_history_ids


print("✅ Conversation function ready!")

✅ Conversation function ready!


## Step 5: Test in Terminal (Without UI)

Let's verify the bot actually remembers the conversation context.

In [ ]:
# Simulate a real multi-turn conversation
print("=" * 55)
print("       CHATBOT CONVERSATION TEST")
print("=" * 55)

test_conversation = [
    "Hi, I need help with my order.",
    "I ordered a laptop but it hasn't arrived yet.",
    "My order number is 12345.",
    "How long will it take?"
]

history = None  # Start with empty history

for message in test_conversation:
    print(f"\n🧑 You    : {message}")
    response, history = generate_response(message, history)
    print(f"🤖 Bot    : {response}")
    print("-" * 55)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


       CHATBOT CONVERSATION TEST

🧑 You    : Hi, I need help with my order.
🤖 Bot    : PM me your info and list
-------------------------------------------------------

🧑 You    : I ordered a laptop but it hasn't arrived yet.
🤖 Bot    : I can help you if you can wait till tomorrow.
-------------------------------------------------------

🧑 You    : My order number is 12345.
🤖 Bot    : Alrighty, I'm going to send you a PM.
-------------------------------------------------------

🧑 You    : How long will it take?
🤖 Bot    : I don't know, I'm sending the code now.
-------------------------------------------------------


## Step 6: Launch Gradio Chat UI

A clean WhatsApp-style chat interface.
The bot will remember everything said in the conversation!

In [ ]:
# Store conversation history across Gradio turns
conversation_history = {"ids": None}


def chat(user_message, history):
    """
    Gradio-compatible chat function.
    Maintains both Gradio display history and model token history.
    """
    if not user_message.strip():
        return "", history

    # Generate response with memory
    bot_response, conversation_history["ids"] = generate_response(
        user_message,
        conversation_history["ids"]
    )

    # Add to Gradio chat display
    history.append((user_message, bot_response))

    return "", history


def reset_chat():
    """Clears both display and model memory."""
    conversation_history["ids"] = None
    return [], ""


# Build Gradio Interface
with gr.Blocks(title="GenAI Customer Support Chatbot") as demo:

    gr.Markdown("""
    # 🤖 GenAI Customer Support Chatbot
    **Powered by Microsoft DialoGPT | Multi-Turn Conversation**

    This chatbot **remembers your conversation** and replies in context.
    Try asking follow-up questions!
    """)

    chatbot = gr.Chatbot(
        label="Customer Support Chat",
        height=450,
        bubble_full_width=False
    )

    with gr.Row():
        user_input = gr.Textbox(
            placeholder="Type your message... (Press Enter to send)",
            label="",
            scale=5,
            autofocus=True
        )
        send_btn = gr.Button("Send 💬", scale=1, variant="primary")

    clear_btn = gr.Button("🗑️ Clear & Start New Chat", variant="secondary")

    gr.Examples(
        label="💡 Try these example questions:",
        examples=[
            "Hi, I need help with my order.",
            "I want to return a product.",
            "My payment failed. What should I do?",
            "How do I track my shipment?",
            "Can I change my delivery address?"
        ],
        inputs=user_input
    )

    # Event Handlers
    send_btn.click(chat, [user_input, chatbot], [user_input, chatbot])
    user_input.submit(chat, [user_input, chatbot], [user_input, chatbot])
    clear_btn.click(reset_chat, outputs=[chatbot, user_input])


# ▶️ For Jupyter Notebook : share=False
# ▶️ For Google Colab     : change to share=True
demo.launch(share=True)

/tmp/ipykernel_12194/2531894793.py:42: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_12194/2531894793.py:42: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_12194/2531894793.py:42: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 📊 Project Summary

| Component | Details |
|---|---|
| **Model** | microsoft/DialoGPT-medium |
| **Framework** | Hugging Face Transformers |
| **Interface** | Gradio |
| **Key Feature** | Multi-Turn Conversation Memory |
| **Technique** | Prompt Engineering + Context Window |
| **Use Case** | E-commerce Customer Support |
| **Language** | Python |

### 🔍 What This Project Demonstrates
- Loading a **pre-trained conversational model** (DialoGPT)
- **Multi-turn conversation** with chat history memory
- **Token management** to handle long conversations
- **Sampling techniques** (top-k, top-p, temperature) for natural replies
- Clean **Gradio chat UI** for real interaction

---
*Built by Syed Mohammad Shahzeb*